<a href="https://colab.research.google.com/github/Joey-Jireh/eye-of-ra/blob/main/notebooks/week2/week2_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Week 2 Header & Library Imports
# Eye of Ra 👁️ — Week 2: Feature Engineering & Model Training

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✅ Libraries loaded — Week 2 begins")
print("👁️ Eye of Ra — Feature Engineering & Model Training")

# Git: Cell 1 — Week 2 header and imports

✅ Libraries loaded — Week 2 begins
👁️ Eye of Ra — Feature Engineering & Model Training


In [3]:
# Cell 2 — Load master dataset
df = pd.read_csv('/content/eye_of_ra_master_dataset_v3.csv')

print("=" * 50)
print("👁️ EYE OF RA — MASTER DATASET LOADED")
print("=" * 50)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFraud labels:   {df['fraud_label'].sum()}")
print(f"Audit flagged:  {df['audit_flagged'].sum()}")
print(f"Clean:          {(df['fraud_label'] == 0).sum()}")
print(f"Fraud rate:     {df['fraud_label'].mean()*100:.1f}%")
print(f"\nMissing values:\n{df.isnull().sum()}")

# Git: Cell 2 — reload master dataset for Week 2

👁️ EYE OF RA — MASTER DATASET LOADED
Shape: 262 rows x 16 columns

Columns: ['year', 'contract_id', 'entity', 'contract_description', 'investigation_result', 'decision', 'fraud_label', 'supplier', 'is_variation', 'is_sole_source', 'entity_frequency', 'supplier_frequency', 'repeat_supplier', 'entity_standard', 'audit_flagged', 'audit_flag_count']

Fraud labels:   23
Audit flagged:  35
Clean:          239
Fraud rate:     8.8%

Missing values:
year                    0
contract_id             0
entity                  0
contract_description    0
investigation_result    4
decision                4
fraud_label             0
supplier                0
is_variation            0
is_sole_source          0
entity_frequency        0
supplier_frequency      0
repeat_supplier         0
entity_standard         0
audit_flagged           0
audit_flag_count        0
dtype: int64


In [4]:
# Cell 3 — Feature 1: Award Concentration Score
# How dominant is one supplier within an entity?
# High concentration = one supplier winning most contracts = red flag

# Step 1: Count contracts per entity-supplier pair
entity_supplier_counts = df.groupby(['entity_standard', 'supplier']).size().reset_index(name='pair_count')

# Step 2: Total contracts per entity
entity_totals = df.groupby('entity_standard').size().reset_index(name='entity_total')

# Step 3: Merge and compute concentration ratio
concentration = entity_supplier_counts.merge(entity_totals, on='entity_standard')
concentration['award_concentration'] = concentration['pair_count'] / concentration['entity_total']

# Step 4: Map back to main dataframe
df = df.merge(concentration[['entity_standard', 'supplier', 'award_concentration']],
              on=['entity_standard', 'supplier'], how='left')

# Step 5: Review
print("=" * 50)
print("✅ FEATURE 1: Award Concentration Score")
print("=" * 50)
print(f"Range: {df['award_concentration'].min():.3f} — {df['award_concentration'].max():.3f}")
print(f"Mean:  {df['award_concentration'].mean():.3f}")
print(f"\nTop 10 highest concentration contracts:")
print(df[['entity_standard', 'supplier', 'award_concentration', 'fraud_label']]
      .sort_values('award_concentration', ascending=False).head(10).to_string())

# Git: Cell 3 — Feature 1: award concentration score

✅ FEATURE 1: Award Concentration Score
Range: 0.062 — 1.000
Mean:  0.580

Top 10 highest concentration contracts:
                              entity_standard                                  supplier  award_concentration  fraud_label
261  Ghana Geological Survey Authority (GGSA)                   Messrs. Forte Logistics                1.000            0
260            Ghana Maritime Authority (GMA)                Messrs. Kaysens Gaisie Ltd                1.000            0
259  Ghana Export Promotion Authority (GEPA))       Messrs. Insight Advertising Limited                1.000            0
243       National Blood Service Ghana (NBSG)  Messrs. Investrade International Co. Ltd                1.000            0
241        Kumasi Metropolitan Assembly (KMA)                                   Unknown                1.000            0
236            Ghana Commodity Exchange (GCX)                         Messrs. DESS Inc.                1.000            0
240        Centre for Plant Medi

In [5]:
# Cell 4 — Feature 2: Method Abuse Score
# Sole-source frequency per entity — entities that repeatedly bypass competitive bidding

# Step 1: Sole source rate per entity
sole_source_rate = df.groupby('entity_standard').agg(
    total_contracts=('contract_id', 'count'),
    sole_source_count=('is_sole_source', 'sum')
).reset_index()

sole_source_rate['method_abuse_score'] = (
    sole_source_rate['sole_source_count'] / sole_source_rate['total_contracts']
)

# Step 2: Map back
df = df.merge(sole_source_rate[['entity_standard', 'method_abuse_score']],
              on='entity_standard', how='left')

# Step 3: Check correlation with fraud
print("=" * 50)
print("✅ FEATURE 2: Method Abuse Score")
print("=" * 50)
print(f"Range: {df['method_abuse_score'].min():.3f} — {df['method_abuse_score'].max():.3f}")
print(f"Mean:  {df['method_abuse_score'].mean():.3f}")

print(f"\nMethod abuse score — fraud vs clean:")
print(df.groupby('fraud_label')['method_abuse_score'].describe().round(3))

print(f"\nTop 10 most abusive entities:")
top_abuse = sole_source_rate.sort_values('method_abuse_score', ascending=False).head(10)
print(top_abuse.to_string())

# Git: Cell 4 — Feature 2: method abuse score

✅ FEATURE 2: Method Abuse Score
Range: 0.000 — 1.000
Mean:  0.179

Method abuse score — fraud vs clean:
              count  mean   std   min   25%   50%   75%   max
fraud_label                                                  
0           239.000 0.182 0.280 0.000 0.000 0.100 0.222 1.000
1            23.000 0.151 0.168 0.000 0.000 0.125 0.244 0.667

Top 10 most abusive entities:
                                                   entity_standard  total_contracts  sole_source_count  method_abuse_score
7       Bulk Oil Storage and Transportation company Limited (BOST)                1                  1               1.000
5              Bulk Oil Storage Transportation Company Ltd. (BOST)                2                  2               1.000
12           Controller and Accountant General’s Department (CAGD)                1                  1               1.000
9                           Cape Coast Technical University (CCTU)                1                  1               1.000
13

In [6]:
# Cell 5 — Feature 3: Supplier Risk Score
# A supplier's historical fraud rate — past behaviour predicts future behaviour

# Step 1: Fraud rate per supplier
supplier_risk = df.groupby('supplier').agg(
    supplier_total=('contract_id', 'count'),
    supplier_fraud_count=('fraud_label', 'sum')
).reset_index()

supplier_risk['supplier_risk_score'] = (
    supplier_risk['supplier_fraud_count'] / supplier_risk['supplier_total']
)

# Step 2: Map back
df = df.merge(supplier_risk[['supplier', 'supplier_risk_score']],
              on='supplier', how='left')

# Step 3: Validate
print("=" * 50)
print("✅ FEATURE 3: Supplier Risk Score")
print("=" * 50)
print(f"Range: {df['supplier_risk_score'].min():.3f} — {df['supplier_risk_score'].max():.3f}")
print(f"Mean:  {df['supplier_risk_score'].mean():.3f}")

print(f"\nSupplier risk score — fraud vs clean:")
print(df.groupby('fraud_label')['supplier_risk_score'].describe().round(3))

print(f"\nHigh-risk suppliers (fraud rate > 0):")
risky = supplier_risk[supplier_risk['supplier_fraud_count'] > 0].sort_values('supplier_risk_score', ascending=False)
print(risky.to_string())

# Git: Cell 5 — Feature 3: supplier risk score

✅ FEATURE 3: Supplier Risk Score
Range: 0.000 — 1.000
Mean:  0.088

Supplier risk score — fraud vs clean:
              count  mean   std   min   25%   50%   75%   max
fraud_label                                                  
0           239.000 0.027 0.033 0.000 0.000 0.000 0.067 0.067
1            23.000 0.716 0.439 0.067 0.067 1.000 1.000 1.000

High-risk suppliers (fraud rate > 0):
                                                                                                                                                supplier  supplier_total  supplier_fraud_count  supplier_risk_score
6                                                                                                                                     EPP Books Services               1                     1                1.000
9    Ghana Institute of Management and Public Administration (GIMPA) to undertake a Labour Rationalization Exercise at the Ghana Railway Company Limited               1               